In [1]:
"""
의학 텍스트 분류기와 프롬프트를 통합한 파일 (GPT API 형식)
"""
import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
from tenacity import retry, stop_after_attempt, wait_exponential
import re
from prompts.medical_prompts import CCPrompts
from prompts.medical_prompts import TreatmentPrompts
from prompts.medical_prompts import TherapyPrompts
# from prompts.medical_prompts import PresentIllnessPrompts

from prompts.PI_prompts import PresentIllnessPrompts_ver2
from prompts.medical_prompts import NumericPrompts

from rate_limiter.rate_limiter import RateLimiter

from tqdm.asyncio import tqdm as tqdm_asyncio

/Users/nam-yeong/git/prj_centum/gpt_word/prompts/medical_prompts.py:487: SyntaxWarning: invalid escape sequence '\ '
  """


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [3]:
df = pd.read_excel('../../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../../data/info.csv')
api_key = api.loc[1][1]

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_61429/962184981.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  api_key = api.loc[1][1]


In [4]:
df.columns = df.columns.str.strip()
df = df[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI']]

In [ ]:
df = df.sample(100)
# df = df.iloc[[4581,16565]]
df

In [19]:
# tst = df.index
tst

Index([10456, 13176, 27860,  5550,  1965,  5693,  1217,  5416, 11918,  7052,
       23021,  1279, 20473,   964, 15255, 12647, 21898, 24004, 21644,  7092,
       16864,  8041, 22474, 11219, 21370, 16400, 20410, 22642, 22646,    15,
        2014, 16103,  5308, 10501, 13638, 18044,  6412, 11842, 11457,  9244,
       27466,  2201, 19473, 10868,  9870, 16037, 11764, 23617,  1439, 18418,
       16983, 23078, 13077,  5916,  6650, 26379, 15917, 27538, 12235, 16722,
       14093,  7382, 13738,  1079, 21726, 19921,  4672, 26050,  6019, 12597,
       18911,  6916,  8744,  2971, 11599,  6863, 18326,  7822,  4176, 16541,
       11727, 24261,  2511, 20792, 25828, 24799, 12668, 26712, 13674, 21260,
       26303, 25958, 27151, 16105, 22932,  2403, 22687,  9379, 21130,  8247],
      dtype='int64')

In [5]:
# 여러 인덱스를 선택할 때는 리스트 형태로 전달해야 합니다
df = df.loc[[13674, 21260, 26303]]

In [6]:
class Config:
    # 실제 OpenAI API 키로 교체하거나 환경변수로부터 로드하세요.
    API_KEY = api_key
    # MODEL_NAME = "gpt-4o"
    MODEL_NAME = "o3-mini"
    MAX_TOKENS = 4096
    TEMPERATURE = 0
    BATCH_SIZE = 100
    SEMAPHORE_LIMIT = 5
    MAX_RETRIES = 4
    LOG_FILE = "medical_classifier.log"
    # RPM_LIMIT = 4000  # 실제 한도보다 약간 낮게 설정
    # TPM_LIMIT = 3800000  # 실제 한도보다 약간 낮게 설정
    
    # o1 모델 API 제한 반영 (약간의 여유를 둠)
    RPM_LIMIT = 4800  # 5,000 RPM
    TPM_LIMIT = 3800000  # 4,000,000 TPM
    TPD_LIMIT = 38000000  # 40,000,000 TPD
    INDEX_COLUMNS = ['환자번호', '날짜']

#############################################
# 로깅 설정 함수
#############################################

def setup_logging(log_file=Config.LOG_FILE):
    """로깅 설정을 초기화하는 함수"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

# 초기 로거 생성 (설정은 아직 적용되지 않음)
logger = logging.getLogger(__name__)

#############################################
# 체크포인트 관리 클래스
#############################################



#############################################
# 메디컬 텍스트 분류기 클래스 (GPT API 사용)
#############################################

class MedicalTextClassifier:
    """의학 텍스트 분류기 - 체크포인트 없이 동작하도록 수정"""
    
    def __init__(self, api_key: str, config=None):
        self.config = config if config is not None else Config
        
        # API 키 설정
        import os
        os.environ["OPENAI_API_KEY"] = api_key
        self.client = openai.OpenAI(api_key=api_key)
        
        self.semaphore = asyncio.Semaphore(self.config.SEMAPHORE_LIMIT)
        
        # Rate Limiter
        self.rate_limiter = RateLimiter(
            rpm_limit=self.config.RPM_LIMIT,
            tpm_limit=self.config.TPM_LIMIT
        )
        
        # 처리 대상 컬럼별 분류 함수
        self.classifiers = {
            'CC': self._classify_cc,
            '약': self._classify_medication,
            '장치': self._classify_device,
            '습관': self._classify_habit,
            '찜질': self._classify_hot_pack,
            '마사지, 스트레칭': self._classify_massage,
        }
   
    
    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """모든 컬럼 처리 - 체크포인트 없이"""
        original_shape = df.shape
        processed_cols = 0
        
        logger.info(f"원본 DataFrame 인덱스 샘플: {df.index.tolist()[:5]}")
        
        # 원본 인덱스 보존
        original_index = df.index.copy()
        original_df = df.reset_index().copy()
        
        # 문자열 형태로 별도 컬럼에 보존 (debug)
        original_df['orig_index'] = original_index.map(str)
        logger.info(f"orig_index 샘플(5): {original_df['orig_index'].tolist()[:5]}")
        
        # 최종 결과를 담을 DataFrame 초기화
        result_df = original_df.copy()
        
        for column in self.classifiers.keys():
            if column not in df.columns:
                continue
            
            logger.info(f"Processing column: {column}")
            column_results = await self._process_column_no_checkpoint(result_df, column)
            
            if column_results is not None and not column_results.empty:
                # column_results에는 orig_index가 인덱스로 설정되어 있음
                derived_cols = [col for col in column_results.columns if col.startswith(f"{column}_")]
                if derived_cols:
                    logger.info(f"[{column}]에서 {len(derived_cols)}개 파생 컬럼 생성: {derived_cols}")
                    
                    # 원본 result_df에 병합
                    for derived_col in derived_cols:
                        # Series를 딕셔너리 형태로 변환 후, orig_index 매핑
                        series_to_join = column_results[derived_col]
                        result_df[derived_col] = result_df['orig_index'].map(series_to_join.to_dict())
                    
                    # 파생 컬럼 내용 검사
                    for derived_col in derived_cols:
                        cnt_nonempty = result_df[derived_col].notna().sum()
                        logger.info(f"  -> {derived_col} - 유효 데이터: {cnt_nonempty}/{len(result_df)}")
            
            processed_cols += 1
        
        # 원래 인덱스로 복원
        if 'index' in result_df.columns:
            result_df.drop('index', axis=1, inplace=True, errors='ignore')
        if 'orig_index' in result_df.columns:
            result_df.drop('orig_index', axis=1, inplace=True)
        
        result_df.index = original_index
        
        logger.info(f"[process_all_columns] 총 {processed_cols}개 컬럼 처리 완료.")
        return result_df

    async def _process_column_no_checkpoint(self, df: pd.DataFrame, column: str) -> pd.DataFrame:
        """체크포인트 없이 단일 컬럼 처리"""
        try:
            mask = df[column].notna() & df[column].astype(str).str.strip().ne('')
            if not mask.any():
                logger.info(f"[{column}] 파싱 대상 텍스트가 없음")
                return pd.DataFrame()
            
            filtered_df = df.loc[mask].copy()
            logger.info(f"[{column}] 처리 대상 행 수: {len(filtered_df)}")
            
            texts_with_idx = [
                (idx, text, orig_idx)
                for idx, text, orig_idx in zip(filtered_df.index,
                                               filtered_df[column],
                                               filtered_df['orig_index'])
            ]
            
            results = await self._safe_process_batches(
                texts=[t for (_, t, _) in texts_with_idx],
                original_indices=[i for (i, _, _) in texts_with_idx],
                orig_indices=[o for (_, _, o) in texts_with_idx],
                classifier=self.classifiers[column],
                column=column
            )
            
            if not results:
                return pd.DataFrame()
            
            results_df = pd.DataFrame(results)
            
            # 'orig_index'를 인덱스로
            if 'orig_index' in results_df.columns:
                results_df.set_index('orig_index', inplace=True)
            
            # 컬럼명 접두사 붙이기
            new_cols = []
            for col_name in results_df.columns:
                if col_name == 'index':  
                    new_cols.append(col_name)  # 'index' 유지
                else:
                    new_cols.append(f"{column}_{col_name}")
            
            results_df.columns = new_cols
            return results_df
        
        except Exception as e:
            logger.error(f"[{column}] 처리 중 오류: {str(e)}")
            return pd.DataFrame()


    async def _safe_process_batches(self, texts: List[str],
                                    original_indices: List[int],
                                    orig_indices: List[str],
                                    classifier, column: str) -> List[Dict]:
        """API 호출 - 체크포인트 없이 디버깅 로그 강화"""
        results = []
        
        for i, (txt, idx, orig_idx) in enumerate(zip(texts, original_indices, orig_indices)):
            # 디버깅: 각 텍스트 출력 (길면 앞부분만)
            logger.info(f"[{column}] {i+1}/{len(texts)} | index={idx}, orig_index={orig_idx}, text={txt[:80]}...")
            
            try:
                if not txt or not str(txt).strip():
                    results.append({"index": idx, "orig_index": orig_idx})
                    continue
                
                # 단일 텍스트 호출
                single_result = await classifier([txt], self.semaphore)
                
                # 응답 확인
                if single_result and len(single_result) > 0:
                    item = single_result[0]
                    merged = {"index": idx, "orig_index": orig_idx}
                    if isinstance(item, dict):
                        merged.update(item)
                    results.append(merged)
                else:
                    results.append({"index": idx, "orig_index": orig_idx})
                
            except Exception as e:
                logger.error(f"[{column}] 인덱스 {idx} 텍스트 처리 오류: {str(e)}")
                results.append({"index": idx, "orig_index": orig_idx})
        
        return results

    @retry(stop=stop_after_attempt(3),
           wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
        """API 직접 호출 - checkpoint 없음"""
        async with semaphore:
            import httpx
            url = "https://api.openai.com/v1/chat/completions"
            headers = {
                "Authorization": f"Bearer {self.config.API_KEY}",
                "Content-Type": "application/json"
            }
            data = {
                "model": "gpt-3.5-turbo",
                "messages": [
                    {"role": "system", "content": "JSON 형식으로 응답하세요."},
                    {"role": "user", "content": prompt}
                ],
                "max_tokens": self.config.MAX_TOKENS,
                "temperature": self.config.TEMPERATURE
            }
            
            response = await httpx.AsyncClient().post(url, headers=headers, json=data, timeout=60.0)
            response.raise_for_status()
            
            response_data = response.json()
            content = response_data["choices"][0]["message"]["content"]
            
            total_tokens = response_data["usage"]["total_tokens"]
            self.rate_limiter.record_request(total_tokens)
            
            # JSON 파싱
            result = self._validate_and_parse_json(content)
            if not result:
                raise ValueError("Invalid JSON structure")
            return result

    def _validate_and_parse_json(self, content: str) -> List[Dict]:
        """JSON 파싱 + 디버깅"""
        logger.info(f"원시 API 응답(앞200자): {content[:200]}...")
        try:
            # 직접 파싱
            parsed = json.loads(content)
            if isinstance(parsed, list):
                return parsed
        except:
            pass
        
        # 정규식 추출 (생략하거나 기존 로직 유지)
        pattern = r'```json\s*([\s\S]*?)```|(\[[\s\S]*\])'
        matches = re.findall(pattern, content)
        for match in matches:
            for m in match:
                m_strip = m.strip()
                if not m_strip:
                    continue
                try:
                    p = json.loads(m_strip)
                    if isinstance(p, list):
                        return p
                except:
                    pass
        
        logger.error("JSON 파싱 실패(할루시네이션 가능)")
        return []
    
    #o1-mini 모델 사용
    # @retry(stop=stop_after_attempt(3),
    # wait=wait_exponential(multiplier=1, min=2, max=10))
    # async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
    #     """API 호출 메소드 (직접 API 호출)"""
    #     try:
    #         async with semaphore:
    #             import httpx
                
    #             # API 요청 준비
    #             url = "https://api.openai.com/v1/chat/completions"
    #             headers = {
    #                 "Authorization": f"Bearer {self.config.API_KEY}",
    #                 "Content-Type": "application/json"
    #             }
    #             data = {
    #                 # "model": "gpt-3.5-turbo",
    #                 "model": self.config.MODEL_NAME,
    #                 "messages": [
    #                     {"role": "system", "content": "JSON 형식으로 응답하세요."},
    #                     {"role": "user", "content": prompt}
    #                 ],
    #                 "max_completion_tokens": self.config.MAX_TOKENS,
    #                 "temperature": self.config.TEMPERATURE
    #             }
                
    #             # API 호출
    #             async with httpx.AsyncClient() as client:
    #                 response = await client.post(url, headers=headers, json=data, timeout=60.0)
    #                 response.raise_for_status()  # 오류 발생 시 예외 발생
    #                 response_data = response.json()
                
    #             # 응답 처리
    #             content = response_data["choices"][0]["message"]["content"]
    #             logger.debug(f"API Response: {content[:200]}...")
                
    #             # 토큰 사용량 기록
    #             total_tokens = response_data["usage"]["total_tokens"]
    #             self.rate_limiter.record_request(total_tokens)
                
    #             # JSON 파싱
    #             result = self._validate_and_parse_json(content)
    #             if not result:
    #                 raise ValueError("Invalid JSON structure")
    #             return result

    #     except httpx.HTTPStatusError as e:
    #         logger.error(f"HTTP error: {e.response.status_code} - {e.response.text}")
    #         raise
    #     except Exception as e:
    #         logger.error(f"API call failed: {str(e)}")
    #         raise

    async def _classify_cc(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        cc_results = await self._make_api_call(CCPrompts.cc_analysis_prompt(texts), semaphore)
        history_results = await self._make_api_call(CCPrompts.cc_history_prompt(texts), semaphore)
        severity_results = await self._make_api_call(CCPrompts.cc_severity_prompt(texts), semaphore)
        
        combined = []
        for i in range(len(texts)):
            merged = {}
            if i < len(cc_results): merged.update(cc_results[i])
            if i < len(history_results): merged.update(history_results[i])
            if i < len(severity_results): merged.update(severity_results[i])
            combined.append(merged)
        return combined

    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        return await self._make_api_call(TreatmentPrompts.medication_prompt(texts), semaphore)

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        return await self._make_api_call(TreatmentPrompts.device_prompt(texts), semaphore)
    
    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        return await self._make_api_call(TreatmentPrompts.habit_prompt(texts), semaphore)

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        return await self._make_api_call(TherapyPrompts.hot_pack_prompt(texts), semaphore)

    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        return await self._make_api_call(TherapyPrompts.massage_prompt(texts), semaphore)
        


#############################################
# 메디컬 데이터 처리 함수
#############################################

async def process_medical_data(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """체크포인트 없이, 디버깅 로그를 포함해 처리"""
    classifier = MedicalTextClassifier(api_key)
    start_time = datetime.now()
    logger.info(f"Starting processing at {start_time}")
    
    processed_df = await classifier.process_all_columns(df)
    
    end_time = datetime.now()
    logger.info(f"Finished processing at {end_time}, duration={end_time - start_time}")
    return processed_df


#############################################
# 메인 함수
#############################################
def main():
    global logger
    logger = setup_logging()
    
    try:
        # 가정: df 라는 이미 로드된 pd.DataFrame
        original_df = df  
        loop = asyncio.get_event_loop()
        
        result_df = loop.run_until_complete(process_medical_data(original_df, Config.API_KEY))
        
        # 결과 저장 등 처리
        result_df.to_csv("final_result_no_checkpoint.csv", index=True, encoding="utf-8-sig")
        logger.info("[main] 최종 결과 CSV 저장 완료")
    
    except Exception as e:
        logger.error(f"[main] 오류 발생: {str(e)}", exc_info=True)


if __name__ == "__main__":
    main()

2025-03-30 22:35:32,152 - __main__ - INFO - Starting processing at 2025-03-30 22:35:32.152697
2025-03-30 22:35:32,153 - __main__ - INFO - 원본 DataFrame 인덱스 샘플: [13674, 21260, 26303]
2025-03-30 22:35:32,156 - __main__ - INFO - orig_index 샘플(5): ['13674', '21260', '26303']
2025-03-30 22:35:32,157 - __main__ - INFO - Processing column: CC
2025-03-30 22:35:32,235 - __main__ - INFO - [CC] 처리 대상 행 수: 3
2025-03-30 22:35:32,236 - __main__ - INFO - [CC] 1/3 | index=0, orig_index=13674, text=구강내과#2[도착]대기고지함)물리치료 , APS del , MMTT 필요여부 CK (PT 원하심)-> 보톡스는 나중에 놔주신다고 했던 것 같아요...
2025-03-30 22:35:33,761 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-03-30 22:35:33,766 - __main__ - INFO - 원시 API 응답(앞200자): [
    {
        "location": "입",
        "pain_type": "통증",
        "painUncomp_desc_jaw": "",
        "disable_desc_jaw": "입을 벌릴 때 통증",
        "muscle_joint_desc_stress": ""
    }
]...
2025-03-30 22:35:35,524 - httpx - INFO - HTTP Request: POST 

In [56]:
df.loc[[4581]]

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI
4581,2305-130,2023-11-20,"물리치료 , 장치 ck구강내과#7증상: 저번이랑 비슷하게 주 1회정도 뻐근함이 있어요, 지금은 입가쪽으로 위치가 변했어요. 보통 인식을 못할때는 괜찮은데 한번 인식을 하면 쭉 그래요,",NaN,"장치: 주 1회 착용, 밴드O, 장치 불편감 : 여전히 아랫니가 아파요.","습관: 딱딱하고 질긴 음식 피하고 있어요, 치아끼리 닿지 않도록 힘풀려고 노력해요.",찜질: 격일로 온찜질 찜질팩으로 식을때까지,"마사지,스트레칭: 매일 마사지만 해요, 스트레칭은 안해요.",* Crowding*치아마모


In [58]:
pd.read_parquet("/Users/nam-yeong/git/prj_centum/gpt_word/checkpoints/CC_checkpoint.parquet")

,index,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,CC_treat_plan,CC_severity,CC_vas,CC_duration
orig_index,,,,,,,,,,,,,
4581,0,입가쪽,뻐근함,턱 통증,턱 관절의 제한된 개구,스트레스로 인한 턱 근육의 긴장,물리치료,"턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기","뻐근함이 있어요, 위치가 변했어요, 인식을 못할때는 괜찮은데 한번 인식을 하면 쭉 그래요",물리치료,None,7,None


In [17]:
original_index = df.index.copy()
original_index
original_df = df.reset_index()
original_df['orig_index'] = original_index.map(str)
original_index = df.index.copy()
original_df = df.reset_index().copy()  # 인덱스를 컬럼으로 변환하여 복사

# 원본 인덱스를 문자열로 보존
original_df['orig_index'] = original_index.map(str)
original_df
result_df = original_df.copy()
result_df

classifier = MedicalTextClassifier(api_key)
classifier.process_all_columns(df)

In [47]:
pd.read_parquet("/Users/nam-yeong/git/prj_centum/gpt_word/checkpoints/CC_checkpoint.parquet")

,index,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,CC_treat_plan,CC_severity,CC_vas,CC_duration
orig_index,,,,,,,,,,,,,
0,0,턱,통증,통증-전혀 없었어요,,스트레칭-10번씩 거의 매일 했어요,물리치료,"턱관절장애 관련 과거 병력, 치료 이력",딱딱하고 질긴거 그냥 먹었어요. 치아끼리 안물려고 노력했어요.,물리치료,NaN,None,None
1,1,"왼쪽 어금니, 위 앞니, 아랫니","딱딱한 소리, 귓속 통증, 어금니 통증, 앞니 금이, 아랫니 깎임","딱 소리, 귓속 통증, 어금니 통증","앞니 금이, 아랫니 깎임",운동 시 더 아픔,보톡스,"왼쪽에서 딱 소리나고(최근), 양쪽 다 음식먹을때, 입벌릴때 귓속이 아파요. 오른쪽...",질기고 딱딱한거 많이 피하지는 않았고 치아 물지 않으려고 했어요.,보톡스,2.0,None,None


### merge

In [23]:
sens = pd.read_parquet('processed_medical_data_20250310_023712.parquet')
nums = pd.read_parquet('../../data/centum_data_numeric_cleaned.parquet')

In [24]:
len(nums), len(sens)


(28108, 28108)

In [25]:
nums.columns

Index(['환자번호', '날짜', 'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획',
       'End feel', 'CMO_before', 'CMO_after', 'MMO_before', 'MMO_after',
       'deviation_pattern_type', 'deviation_direction', 'deviation_intensity',
       'Cap.pal_Pain_Intensity', 'Cap.pal_Pain_Direction',
       'Cap.pal_Pain_Situation', 'M.pal_Pain_Intensity',
       'M.pal_Pain_Direction', 'M.pal_Pain_Situation', 'Noise_Code',
       'Noise_Direction', 'Noise_Intensity', 'Noise_Situation',
       'Occlusion_lt_number', 'Occlusion_rt_number', 'Occlusion_lt_Intensity',
       'Occlusion_rt_Intensity', 'oj', 'ob', 'Midline_Shift_Jaw',
       'Midline_Shift_Direction_x', 'Midline_Shift_Direction_y',
       'Midline_Shift_Amount', 'CRCO_Direction_x', 'CRCO_Direction_y',
       'CRCO_Amount', 'Tongue_ridging_Intensity', 'Rt_before', 'Rt_after',
       'Lt_bef

In [26]:
sens = sens[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method']]

In [27]:
fin_df = pd.merge(nums, sens, on=['환자번호','날짜'], how='left')
cols = [
       '환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',
       'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획',
       'End feel',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method',
       'CMO_before', 'CMO_after', 'MMO_before', 'MMO_after',
       'deviation_pattern_type', 'deviation_direction', 'deviation_intensity',
       'Cap.pal_Pain_Intensity', 'Cap.pal_Pain_Direction',
       'Cap.pal_Pain_Situation', 'M.pal_Pain_Intensity',
       'M.pal_Pain_Direction', 'M.pal_Pain_Situation', 'Noise_Code',
       'Noise_Direction', 'Noise_Intensity', 'Noise_Situation',
       'Occlusion_lt_number', 'Occlusion_rt_number', 'Occlusion_lt_Intensity',
       'Occlusion_rt_Intensity', 'oj', 'ob', 'Midline_Shift_Jaw',
       'Midline_Shift_Direction_x', 'Midline_Shift_Direction_y',
       'Midline_Shift_Amount', 'CRCO_Direction_x', 'CRCO_Direction_y',
       'CRCO_Amount', 'Tongue_ridging_Intensity', 'Rt_before', 'Rt_after',
       'Lt_before', 'Lt_after', 'Next_Visit_Days'
]
fin_df = fin_df[cols]
fin_df.head()

fin_df.to_parquet('../../data/final_without_pi_centum_data_with_medical_data.parquet')
fin_df.to_csv('../../data/final_without_pi_centum_data_with_medical_data.csv', index=False, encoding='utf-8-sig')